# 82514 · Sesión S20 — Plan B: ROS 2 Jazzy en Colab, sin ventanas

**Bloque 7** · jueves 26 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** Para quien no tenga ROS 2 en su portátil: instala ROS 2 Jazzy en la máquina de Colab y hace, sin ventanas, la comprobación de S20 (dos nodos que se hablan y dos que no, por la QoS), un nodo del curso con un parámetro, un servicio y una acción, la edad del dato según la cola, el paquete del almacén construido con colcon, el grafo dibujado en lugar de rqt_graph y, para una pareja sin portátil, las dos rondas del taller de S21. RViz y rqt_graph no se pueden: no hay pantalla.

**Se apoya en:** la documentación de ROS 2 Jazzy (nodos, topics, servicios, acciones, parámetros y calidad de servicio); el paquete curso_almacen del curso.

**Cómo usarlo en clase.** Sigue el guion de la sesión S20 en las diapositivas del bloque 7. Ejecuta la celda de instalación una sola vez al empezar; en Colab tarda unos dos minutos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

> **Aviso para Colab.** La celda de instalación pone ROS 2 Jazzy (sin ventanas) en la máquina de Colab, desde la instantánea del 5 de octubre de 2026: tarda unos dos minutos y hay que repetirla cada vez que Colab da una máquina nueva.
>
> **En clase (jueves 26):** solo si no tienes ROS 2 en tu portátil. Ejecuta la instalación al empezar la comprobación y haz las secciones 1 y 2, que son la comprobación (1). En el último cuarto de hora, la sección 6, que es la (2) sin ventanas: construye y lanza el mismo paquete (RViz no se puede aquí).
>
> **En casa:** las secciones 3 a 7 y los ejercicios; las soluciones están al final.
>
> **El lunes 30 (S21):** si vuestra pareja no tiene ningún portátil con ROS 2, hacéis aquí las dos rondas del taller (sección 8), con `taller_s21.zip` de Moodle, y en el zip va `grafo.png` en lugar de la captura de `rqt_graph`. Colab os dará una máquina nueva: antes de la sección 8, la instalación, la celda de las ayudas y la primera celda de la sección 6.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

In [ ]:
# Ejecutar una sola vez. En Colab tarda unos dos minutos: instala ROS 2 Jazzy (sin ventanas) de la instantánea del
# 5 de octubre de 2026 (rclpy 7.1.12), la misma que pide el documento de instalación del portátil.
import os, sys, time, signal, subprocess, textwrap, base64, io, zipfile, re, glob
import numpy as np, matplotlib
import matplotlib.pyplot as plt

FIJAS = {'numpy': '2.4.6', 'matplotlib': '3.10.0'}          # las de Colab con las que se probó el bloque 7
EN_COLAB = 'google.colab' in sys.modules
SETUP = '/opt/ros/jazzy/setup.bash'
PY = '/usr/bin/python3.12'                                   # el Python del sistema, el de rclpy (el del cuaderno es el 3.13)
INSTANTANEA = 'http://snapshots.ros.org/jazzy/2026-10-05/ubuntu'
CLAVE = 'https://keyserver.ubuntu.com/pks/lookup?op=get&search=0x4B63CF8FDE49746E98FA01DDAD19BAB3CBF125EA'
PAQUETES = ('ros-jazzy-ros-base ros-jazzy-demo-nodes-py ros-jazzy-action-tutorials-py ros-jazzy-tf2-tools '
            'ros-dev-tools graphviz')


def instalar_ros2():
    t0 = time.time()
    if EN_COLAB and not os.path.exists(SETUP):
        print('Instalando ROS 2 Jazzy, sin ventanas: unos dos minutos...')
        subprocess.run(f'curl -sSL "{CLAVE}" | gpg --dearmor > /usr/share/keyrings/ros-snapshots.gpg', shell=True, check=False)
        subprocess.run(f'echo "deb [arch=amd64 signed-by=/usr/share/keyrings/ros-snapshots.gpg] {INSTANTANEA} noble main" > /etc/apt/sources.list.d/ros2.list', shell=True, check=False)
        subprocess.run(f'apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {PAQUETES} > /dev/null', shell=True, check=False)
    print('ROS 2 Jazzy:', 'instalado' if os.path.exists(SETUP) else 'no está (fuera de Colab, este cuaderno no instala nada)',
          f'· {time.time() - t0:.0f} s')


for nombre, mod in (('numpy', np), ('matplotlib', matplotlib)):
    print(nombre, mod.__version__, '' if mod.__version__ == FIJAS[nombre] else f'(probado con {FIJAS[nombre]})')
print('en Colab:', EN_COLAB, '· Python del cuaderno:', sys.version.split()[0], '· Python del sistema (rclpy):',
      'sí' if os.path.exists(PY) else 'NO: este plan B no sirve en esta máquina',
      '· /usr/bin/python3 es', os.path.realpath('/usr/bin/python3'))
instalar_ros2()
HAY_ROS = os.path.exists(SETUP) and os.path.exists(PY)

In [ ]:
# Las ayudas: ros() ejecuta una orden y devuelve lo que imprime; lanzar() deja un proceso de fondo; parar() lo para;
# grafo() dibuja el grafo de nodos y topics en grafo.png (secciones 7 y 8).
# Todo va con el Python del sistema (rclpy) y con el descubrimiento limitado a esta máquina, como en el aula.
# El workspace de colcon: en Colab, en /content (se ve en el panel de archivos de la izquierda); fuera, en ~/ws_curso.
WS = '/content/ws_curso' if EN_COLAB else os.path.expanduser('~/ws_curso')
ENTORNO = (f'export AMENT_PYTHON_EXECUTABLE={PY} && source {SETUP} && '
           'export ROS_AUTOMATIC_DISCOVERY_RANGE=LOCALHOST ROS_DOMAIN_ID=0 PYTHONUNBUFFERED=1 RCUTILS_LOGGING_BUFFERED_STREAM=0 && '
           f'{{ [ -f {WS}/install/setup.bash ] && source {WS}/install/setup.bash; true; }} && ')
PAQ_PY = ('demo_nodes_py', 'action_tutorials_py', 'tf2_tools', 'curso_almacen', 'taller_s21')   # ejecutables de Python


def _orden(o):
    # Todo con el Python del sistema, aunque python3 sea otro: colcon construye con él (y los ejecutables de los paquetes
    # del curso salen con #!/usr/bin/python3.12), ros2 run lo antepone a los nodos de Python y la CLI ros2 corre con él.
    if o.startswith('colcon '):
        return f'{PY} /usr/bin/colcon ' + o[len('colcon '):]
    if o.startswith('ros2 run ') and o.split()[2] in PAQ_PY:
        o = o.replace('ros2 run ', f'ros2 run --prefix {PY} ', 1)
    return o.replace('ros2 ', f'{PY} /opt/ros/jazzy/bin/ros2 ', 1) if o.startswith('ros2 ') else o


def ros(orden, segundos=10, carpeta=None):
    """Una orden de ROS 2 ('ros2 node list', 'ros2 topic echo --once /cmd_vel'...), en la carpeta que se diga:
    devuelve lo que imprime."""
    if not HAY_ROS:
        return '(sin ROS 2 en esta máquina)'
    r = subprocess.run(['bash', '-c', ENTORNO + f'timeout {segundos} ' + _orden(orden)], capture_output=True, text=True,
                       cwd=carpeta)
    return (r.stdout + r.stderr).strip()


def lanzar(orden):
    """Un proceso de fondo (un nodo, un launch); se para con parar(p)."""
    if not HAY_ROS:
        return None
    return subprocess.Popen(['bash', '-c', ENTORNO + _orden(orden)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, start_new_session=True)


def parar(p):
    if p is None:
        return ''
    os.killpg(p.pid, signal.SIGINT)
    try:
        return p.communicate(timeout=8)[0]
    except subprocess.TimeoutExpired:
        os.killpg(p.pid, signal.SIGKILL)
        return p.communicate()[0]


def lineas(texto, clave, n=4):
    """Las n primeras líneas de texto que contienen clave."""
    return [l.strip() for l in texto.splitlines() if clave in l][:n]


def grafo(fichero='grafo.png'):
    """El grafo de nodos y topics que hay ahora mismo, dibujado con graphviz (dot)."""
    if not HAY_ROS:
        return '(sin ROS 2 en esta máquina)'
    aristas, topics = set(), set()
    for nodo in [n for n in ros('ros2 node list').split() if n.startswith('/')]:
        info, parte = ros(f'ros2 node info {nodo}'), None
        for l in info.splitlines():
            t = l.strip()
            if t.endswith(':') and not t.startswith('/'):
                parte = t[:-1]
            elif t.startswith('/') and parte in ('Subscribers', 'Publishers'):
                top = t.split(':')[0]
                if top in ('/parameter_events', '/rosout'):
                    continue
                topics.add(top)
                aristas.add((top, nodo) if parte == 'Subscribers' else (nodo, top))
    gv = ['digraph G { rankdir=LR; node [fontname=sans];']
    gv += [f'"{t}" [shape=box, style="rounded,filled", fillcolor="#E3F7EC", color="#1FA355"];' for t in sorted(topics)]
    gv += [f'"{n}" [shape=box, style="rounded,filled", fillcolor="#1B2A80", fontcolor=white];'
           for n in sorted({a for a, b in aristas} | {b for a, b in aristas}) if n not in topics]
    gv += [f'"{a}" -> "{b}";' for a, b in sorted(aristas)] + ['}']
    open('grafo.gv', 'w').write('\n'.join(gv))
    r = subprocess.run(['dot', '-Tpng', 'grafo.gv', '-o', fichero], capture_output=True, text=True)
    return f'{fichero}: {len(topics)} topics y {len(aristas)} flechas' + (f' ({r.stderr.strip()})' if r.returncode else '')


print(ros('ros2 --help', 5)[:60] if HAY_ROS else ros(''))

## 1. Dos nodos que se hablan (diapositiva 26)

La comprobación (1), las dos primeras pruebas: `ros2 doctor` tiene que acabar en *All 5 checks passed* con internet, o en *All 3 checks passed* sin él (los avisos *has been updated to a new version* y *Fail to call … Check* son normales: la instantánea fija las versiones y, sin red, no puede compararlas; lo que importa es *passed*, no *failed*), y el `listener` tiene que oír al `talker`.

In [ ]:
doctor = ros('ros2 doctor', 60)
print('ros2 doctor:', lineas(doctor, 'passed', 1) or doctor[-200:])

talker = lanzar('ros2 run demo_nodes_py talker')
time.sleep(1.5)
oye = ros('ros2 run demo_nodes_py listener', 5)
parar(talker)
print('\n'.join(lineas(oye, 'I heard', 4)) or oye)

## 2. Dos que no: la QoS incompatible (diapositivas 14 y 26)

Un publicador **al mejor esfuerzo** y un suscriptor que pide **fiable**: no conectan. Tiene que salir el aviso *offering incompatible QoS* y ningún mensaje. `ros2 topic info -v` enseña la QoS de los dos extremos. Sin opciones, `ros2 topic echo` se adapta al publicador y sí conecta.

In [ ]:
pub = lanzar('ros2 topic pub -r 10 --qos-reliability best_effort /prueba std_msgs/msg/String "{data: hola}"')
time.sleep(2)
eco = lanzar('ros2 topic echo --qos-reliability reliable /prueba')
time.sleep(3)
info = ros('ros2 topic info -v /prueba', 6)
eco_txt = parar(eco)
print('el suscriptor fiable recibe', eco_txt.count('data: hola'), 'mensajes')
print('\n'.join(lineas(eco_txt, 'incompatible', 1)) or eco_txt[-300:])
print('\n'.join(lineas(info, 'Endpoint type', 2) + lineas(info, 'Reliability', 2)))
sin_opciones = ros('ros2 topic echo --once /prueba', 6)
parar(pub)
print('sin opciones, ros2 topic echo recibe:', ' '.join(lineas(sin_opciones, 'data:', 1)) or sin_opciones[-200:])

## 3. Un nodo del curso con un parámetro (diapositivas 8 y 21)

Un nodo escrito desde el cuaderno: publica la consigna de velocidad del robot del almacén (`/cmd_vel`, un `Twist`) a 10 Hz y declara el parámetro `v`, la velocidad, 0,5 m/s. Se arranca como proceso aparte, y la terminal lo inspecciona y le cambia el parámetro en marcha.

In [ ]:
NODO = textwrap.dedent('''
    import rclpy
    from rclpy.node import Node
    from geometry_msgs.msg import Twist

    class Conductor(Node):
        def __init__(self):
            super().__init__('conductor')
            self.declare_parameter('v', 0.5)                  # m/s, la del robot del almacén
            self.pub = self.create_publisher(Twist, 'cmd_vel', 10)
            self.create_timer(0.1, self.tic)                  # 10 Hz

        def tic(self):
            m = Twist()
            m.linear.x = self.get_parameter('v').value
            self.pub.publish(m)

    rclpy.init()
    n = Conductor()
    try:
        rclpy.spin(n)
    except KeyboardInterrupt:
        pass
''')
if HAY_ROS:
    open('conductor.py', 'w').write(NODO)
cond = lanzar(f'{PY} conductor.py')
time.sleep(4)
print('nodos:', [n for n in ros('ros2 node list').split() if n.startswith('/')])
print('ritmo:', lineas(ros('ros2 topic hz /cmd_vel', 4), 'average', 1))
param = ros('ros2 param set /conductor v 0.3')
print('parámetro:', ' '.join(lineas(param, 'parameter', 1)) or param[-200:])
msg = ros('ros2 topic echo --once /cmd_vel')
print('un mensaje:', ' '.join(msg[msg.find('linear:'):].split()[:6]) if 'linear:' in msg else msg[-200:])
parar(cond);

## 4. Un servicio y una acción (diapositivas 17 y 18)

Dos servidores de los ejemplos de ROS 2: uno suma dos enteros (un servicio: pregunta y respuesta) y otro calcula la sucesión de Fibonacci poco a poco (una acción: meta, realimentación y resultado).

In [ ]:
srv = lanzar('ros2 run demo_nodes_py add_two_ints_server')
time.sleep(1.5)
r = ros('ros2 service call /add_two_ints example_interfaces/srv/AddTwoInts "{a: 2, b: 3}"')
print('la respuesta:', r.split('response:')[-1].strip()[:60])
parar(srv)

acc = lanzar('ros2 run action_tutorials_py fibonacci_action_server')
time.sleep(1.5)
r = ros('ros2 action send_goal --feedback /fibonacci action_tutorials_interfaces/action/Fibonacci "{order: 5}"', 20)
parar(acc)
print('realimentaciones:', r.count('Feedback:'), '·', lineas(r, 'status', 1))

## 5. La cola decide la edad del dato (diapositiva 13)

Un sensor publica a 10 Hz, con el instante de la medida en la cabecera; un suscriptor tarda 0,25 s con cada mensaje. Con la profundidad por defecto (10), el dato que atiende tiene casi un segundo: (N − ½)·T = 9,5 · 0,1 s = 0,95 s. Con profundidad 1, unos 0,05 s. Tarda unos 30 segundos.

In [ ]:
EDAD = textwrap.dedent('''
    import sys, time, rclpy
    from rclpy.node import Node
    from rclpy.qos import QoSProfile
    from std_msgs.msg import Header

    class Sensor(Node):
        def __init__(self):
            super().__init__('sensor')
            self.pub = self.create_publisher(Header, 'medida', 10)
            self.create_timer(0.1, self.tic)                  # 10 Hz
        def tic(self):
            h = Header(); h.stamp = self.get_clock().now().to_msg(); self.pub.publish(h)

    class Lento(Node):
        def __init__(self, n):
            super().__init__('lento')
            self.edades = []
            self.create_subscription(Header, 'medida', self.retorno, QoSProfile(depth=n))
        def retorno(self, h):
            ahora = self.get_clock().now().nanoseconds * 1e-9
            self.edades.append(ahora - (h.stamp.sec + h.stamp.nanosec * 1e-9))
            time.sleep(0.25)                                   # el trabajo: 0,25 s por mensaje

    rclpy.init()
    if sys.argv[1] == 'sensor':
        n = Sensor()
        try:
            rclpy.spin(n)
        except KeyboardInterrupt:
            pass
    else:
        N, T = int(sys.argv[2]), float(sys.argv[3])
        n = Lento(N); t0 = time.time()
        while time.time() - t0 < T:
            rclpy.spin_once(n, timeout_sec=0.05)
        e = n.edades[len(n.edades) // 4:]
        print(f'{N} {sum(e) / len(e):.3f} {len(n.edades)}')
''')
if HAY_ROS:
    open('edad_cola.py', 'w').write(EDAD)


def edad_media(N, T=12):
    """Arranca el sensor y el suscriptor lento con profundidad N durante T s; devuelve la edad media (s) o None."""
    if not HAY_ROS:
        return None
    s = lanzar(f'{PY} edad_cola.py sensor')
    time.sleep(1.5)
    out = ros(f'{PY} edad_cola.py lento {N} {T}', T + 10)
    parar(s)
    m = re.search(rf'^{N} ([0-9.]+) ', out, re.M)
    return float(m.group(1)) if m else None


for N in (1, 10):
    e = edad_media(N)
    cm = 50 * round(e, 2) if e is not None else 0                      # la distancia a 0,5 m/s, como en la diapositiva 13
    print(f'profundidad {N:2d}: ' + (f'edad media {e:.2f} s'.replace('.', ',') + f' (la cuenta, {(N - 0.5) * 0.1:.2f} s)'.replace('.', ',')
                                     + f'; a 0,5 m/s, {cm:.{1 if cm < 10 else 0}f} cm'.replace('.', ',') if e is not None else '(sin ROS 2 en esta máquina)'))

## 6. El almacén del curso, construido y lanzado sin ventanas (diapositivas 23 y 27)

El paquete `curso_almacen` es el mismo que el de Moodle: la celda lo escribe en el workspace de colcon (`ws_curso/src`, en el panel de archivos), lo construye y lo lanza con `rviz:=false`. Tienen que salir los nodos del simulador y de las transformadas fijas, `/scan` a 10 Hz con la QoS de sensor (`BEST_EFFORT`) y la cadena `map → laser`.

In [ ]:
# curso_almacen.zip, el de Moodle, en base64 (lo pega aquí el generador del cuaderno desde el paquete del curso)
CURSO_ALMACEN = (
    '...'
    'AAAAAA=='
)
PAQUETE = zipfile.ZipFile(io.BytesIO(base64.b64decode(''.join(CURSO_ALMACEN))))
print('curso_almacen.zip:', len(PAQUETE.namelist()), 'ficheros')
if HAY_ROS:
    os.makedirs(os.path.join(WS, 'src'), exist_ok=True)
    PAQUETE.extractall(os.path.join(WS, 'src'))
    print('en src/:', sorted(os.listdir(os.path.join(WS, 'src'))))
print(ros('colcon build --symlink-install', 180, carpeta=WS)[-160:])

In [ ]:
alm = lanzar('ros2 launch curso_almacen almacen.launch.py rviz:=false')
time.sleep(5)
print('nodos:', [n for n in ros('ros2 node list').split() if n.startswith('/')])
print('/scan:', lineas(ros('ros2 topic hz /scan', 8), 'average', 99)[-1:], lineas(ros('ros2 topic info -v /scan'), 'Reliability', 1))   # la última ventana
print('map -> laser:', lineas(ros('ros2 run tf2_ros tf2_echo map laser', 6), 'Translation', 1))

## 7. El grafo, dibujado (en lugar de rqt_graph)

Sin ventanas no hay `rqt_graph`. `grafo()`, en la celda de las ayudas, pregunta a cada nodo qué publica y qué lee (`ros2 node info`) y lo dibuja con graphviz en `grafo.png`, con los nodos en azul y los topics en verde, como en la diapositiva 8. El almacén de la sección 6 sigue en marcha; al acabar, se para.

In [ ]:
print(grafo())
if HAY_ROS and os.path.exists('grafo.png'):
    from IPython.display import Image, display
    display(Image('grafo.png'))
parar(alm);

## 8. El taller del lunes 30, sin portátil (S21)

Solo para una pareja que no tenga **ningún** portátil con ROS 2 que funcione: las dos rondas del taller de S21, aquí, sin ventanas. El almacén es el mismo de la sección 6, un diferencial con LiDAR de la familia del TurtleBot3, así que lo que hagáis vale igual para quien eligió el TurtleBot3.

1. En una máquina nueva de Colab (el lunes 30 lo será), antes: la celda de instalación, la de las ayudas y la primera de la sección 6, que deja `curso_almacen` en `ws_curso/src`. Después, poned vuestro equipo y vuestra pareja en la primera celda de esta sección, bajad `taller_s21.zip` de Moodle y subidlo con esa celda: se descomprime en `ws_curso/src`, junto a `curso_almacen`, y se construye.
2. Abrid en el panel de archivos de la izquierda `ws_curso/src/taller_s21/taller_s21/vigilante.py` (ronda 1) o `ir_a_punto.py` (ronda 2), rellenad los huecos y guardad (Ctrl+S): con `--symlink-install` no hace falta volver a construir.
3. Ejecutad la celda de la ronda. En lugar de la captura de `rqt_graph`, la ronda 1 dibuja `grafo.png`; la ronda 2 escribe `frames.pdf`.
4. Al final, rellenad `ws_curso/src/taller_s21/medidas.txt` en el mismo panel; la última celda hace `S21_E<n><A/B>.zip` (sin `build`, `install` ni `log`) y lo descarga para subirlo a la Tarea «Taller de ROS 2».

In [ ]:
if globals().get('alm') is not None and alm.poll() is None:
    parar(alm)                             # el almacén de la sección 6, si sigue en marcha (sin pasar por la 7)
EQUIPO, PAREJA = 3, 'A'                    # vuestro equipo (1 a 5) y vuestra pareja (A o B)


def subir_taller():
    """taller_s21.zip, el de Moodle: en Colab se sube con el botón que aparece; fuera, se busca en esta carpeta.
    Se descomprime en ws_curso/src y se construye el workspace entero."""
    if not HAY_ROS:
        return '(sin ROS 2 en esta máquina)'
    if not os.path.isdir(os.path.join(WS, 'src', 'curso_almacen')):
        return '(falta curso_almacen en ws_curso/src: antes, la primera celda de la sección 6)'
    if EN_COLAB:
        from google.colab import files
        nombre = next(iter(files.upload()))     # elegid taller_s21.zip
    else:
        nombre = 'taller_s21.zip'
        if not os.path.exists(nombre):
            return '(sin taller_s21.zip en esta carpeta)'
    zipfile.ZipFile(nombre).extractall(os.path.join(WS, 'src'))
    return ros('colcon build --symlink-install', 180, carpeta=WS)[-160:]


print(subir_taller())

In [ ]:
# Ronda 1: el vigilante. El almacén con el robot en (0,6; 2,4), sin ventanas, y el vigilante durante 20 s.
def ronda1():
    if not HAY_ROS or not os.path.isdir(os.path.join(WS, 'src', 'taller_s21')):
        return '(sin ROS 2 o sin taller_s21: la celda de subir, antes)'
    alm = lanzar('ros2 launch curso_almacen almacen.launch.py rviz:=false x0:=0.6 y0:=2.4')
    time.sleep(5)
    vig = lanzar('ros2 run taller_s21 vigilante')
    time.sleep(20)
    print('/scan:', lineas(ros('ros2 topic info -v /scan'), 'Reliability', 2))
    print(grafo('grafo.png'))                 # en lugar de rqt_graph.png
    print('dónde está el robot (la verdad):', lineas(ros('ros2 run tf2_ros tf2_echo map verdad', 6), 'Translation', 1))
    salida = parar(vig)
    parar(alm)
    return 'lo que dice el vigilante:\n' + '\n'.join(salida.strip().splitlines()[-4:])


print(ronda1())

In [ ]:
# Ronda 2: ir a (1,8; 3,6) controlando en odom y después en map, con el almacén recién lanzado cada vez
def viaje(marco, arbol=False):
    """Un viaje del muelle a la meta controlando en «marco»; con arbol=True, además, frames.pdf con view_frames."""
    if not HAY_ROS or not os.path.isdir(os.path.join(WS, 'src', 'taller_s21')):
        return '(sin ROS 2 o sin taller_s21: la celda de subir, antes)'
    alm = lanzar(f'ros2 launch curso_almacen almacen.launch.py rviz:=false semilla:={EQUIPO}')
    time.sleep(5)
    ir = lanzar(f'ros2 launch taller_s21 ir_a_punto.launch.py marco:={marco}')
    time.sleep(22)                           # el viaje dura unos 12,5 s
    if arbol:
        for f in glob.glob('frames_*'):
            os.remove(f)
        ros('ros2 run tf2_tools view_frames', 25)
        pdfs = sorted(glob.glob('frames_*.pdf'))
        if pdfs:
            os.replace(pdfs[-1], 'frames.pdf')
        print('frames.pdf:', 'escrito' if os.path.exists('frames.pdf') else 'no ha salido')
    verdad = lineas(ros('ros2 run tf2_ros tf2_echo map verdad', 6), 'Translation', 1)
    texto = parar(ir)
    parar(alm)
    return '\n'.join(lineas(texto, 'Llegado', 1) or lineas(texto, 'Falta', 1) or texto.strip().splitlines()[-3:]) + f'\nla verdad: {verdad}'


print('controlando en odom:\n' + viaje('odom'))
print('controlando en map:\n' + viaje('map', arbol=True))

In [ ]:
# El zip de la pareja: el paquete sin build, install ni log (con medidas.txt dentro, rellenado), grafo.png y frames.pdf
def zip_pareja():
    src = os.path.join(WS, 'src', 'taller_s21')
    if not HAY_ROS or not os.path.isdir(src):
        return '(sin ROS 2 o sin el paquete taller_s21: la celda de subir, antes)'
    nombre = f'S21_E{EQUIPO}{PAREJA}'
    with zipfile.ZipFile(nombre + '.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for raiz, dirs, fichs in os.walk(src):
            dirs[:] = [d for d in dirs if d not in ('build', 'install', 'log', '__pycache__')]
            for f in fichs:
                p = os.path.join(raiz, f)
                z.write(p, os.path.join(nombre, 'taller_s21', os.path.relpath(p, src)))
        for f in ('grafo.png', 'frames.pdf'):
            if os.path.exists(f):
                z.write(f, os.path.join(nombre, os.path.basename(f)))
        dentro = z.namelist()
    if EN_COLAB:
        from google.colab import files
        files.download(nombre + '.zip')
    return f'{nombre}.zip: {len(dentro)} ficheros: ' + ', '.join(sorted(dentro))


print(zip_pareja())

### Ejercicio 1 (en casa): las otras combinaciones de la QoS

Repite la sección 2 con el suscriptor **al mejor esfuerzo** frente al publicador al mejor esfuerzo, y después con el publicador **fiable** frente al suscriptor al mejor esfuerzo. ¿Conectan? Compara con la tabla de la diapositiva 14.

In [ ]:
# Ejercicio 1: cambia las dos --qos-reliability de la sección 2 y mira si llega algún mensaje.
# pub = lanzar('ros2 topic pub -r 10 --qos-reliability reliable /prueba std_msgs/msg/String "{data: hola}"')
# ...

### Ejercicio 2 (en casa): la profundidad 5

Mide con `edad_media(5)` la edad del dato con profundidad 5 y compárala con la cuenta, (N − ½)·T. ¿Cuántos centímetros son a 0,5 m/s?

In [ ]:
# Ejercicio 2: edad_media(5)

### Ejercicio 3 (en casa): muchos a muchos

Con el `talker` de la sección 1 en marcha, lanza **dos** `listener` a la vez. ¿Oyen los dos lo mismo? ¿Lo sabe el `talker`?

In [ ]:
# Ejercicio 3: talker = lanzar(...); dos listener con lanzar(); parar() los tres y compara lo que oyen.

## Soluciones

Medidas en un Ubuntu 24.04.5 como el de Colab, con ROS 2 Jazzy (rclpy 7.1.12); en Colab, los números cambian poco.

**Ejercicio 1.** Al mejor esfuerzo frente a al mejor esfuerzo, conectan; fiable frente a un suscriptor al mejor esfuerzo, también (lo que pide el suscriptor no exige más que lo que ofrece el publicador). Solo falla la combinación de la sección 2: el suscriptor pide fiable y el publicador ofrece al mejor esfuerzo.

**Ejercicio 2.** Con profundidad 5, unos 0,46 s (la cuenta: 4,5 · 0,1 = 0,45 s), unos 23 cm a 0,5 m/s. Con 1, 0,05 s; con 10, 0,95 s.

**Ejercicio 3.** Los dos listener oyen los mismos mensajes desde que los dos están conectados (los números que oyen a la vez son los mismos), y el talker no sabe cuántos escuchan: publicar es muchos a muchos (diapositiva 9).

In [ ]:
# Solución 1: las dos combinaciones que conectan (echo --once espera al primer mensaje, hasta 10 s)
for p_qos, s_qos in (('best_effort', 'best_effort'), ('reliable', 'best_effort')):
    pub = lanzar(f'ros2 topic pub -r 10 --qos-reliability {p_qos} /prueba std_msgs/msg/String "{{data: hola}}"')
    time.sleep(2)
    r = ros(f'ros2 topic echo --once --qos-reliability {s_qos} /prueba', 10)
    parar(pub)
    print(f'publicador {p_qos}, suscriptor {s_qos}:', 'conectan' if 'data: hola' in r else 'no llega nada en 10 s' if HAY_ROS else r)

In [ ]:
# Solución 2
e5 = edad_media(5)
print('profundidad 5:', f'{e5:.2f} s'.replace('.', ',') if e5 is not None else '(sin ROS 2 en esta máquina)')

In [ ]:
# Solución 3
talker = lanzar('ros2 run demo_nodes_py talker')
time.sleep(1)
l1 = lanzar('ros2 run demo_nodes_py listener')
l2 = lanzar('ros2 run demo_nodes_py listener')
time.sleep(5)
o1, o2 = parar(l1), parar(l2)
parar(talker)
oidos = [set(re.findall(r'Hello World: (\d+)', o)) for o in (o1, o2)]
comunes = sorted(oidos[0] & oidos[1], key=int)
print('listener 1 oye', len(oidos[0]), 'mensajes; listener 2,', len(oidos[1]), '; los dos, los mismos', len(comunes), ':', comunes)

---

## Para llevarse de esta sesión

- **Un robot es un grafo** de procesos que se pasan mensajes: aquí, cada nodo es un proceso aparte con el Python del sistema.
- **La QoS es un contrato:** al mejor esfuerzo frente a fiable no conecta, y solo queda un aviso.
- **Sin ventanas también se comprueba:** `ros2 topic hz`, `topic info -v`, `tf2_echo` y el grafo dibujado sustituyen a RViz y a `rqt_graph`.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*